# Ориентация текстовых кропов: 0° или 180°

Для каждого кропа с текстом предсказываем `p_180` — вероятность того, что текст перевёрнут. Метрика — `1 − Brier score`.

**Подход:** дистилляция. Крупный учитель (ConvNeXt-Tiny) учится отличать ровный текст от перевёрнутого на синтетике и реальных кропах, затем размечает большой пул реальных кропов мягкими вероятностями. На этих вероятностях учится маленький студент (PP-LCNet ×0.5), который и делает предсказания для теста.

Подробное описание подхода, валидации и результатов будет дополняться по ходу работы.

**Разделы**
0. Окружение
1. Скачивание данных — нужно только для обучения

## 0. Окружение

Ячейка ниже создаёт виртуальное окружение `.venv` в корне репозитория, ставит в него зависимости из `requirements.txt` и регистрирует ядро Jupyter «Python (avito-orient)». Её можно запускать из любого ядра с Python 3.10+.

После неё выберите ядро `.venv` (в VS Code: Select Kernel → Python Environments → `.venv`) и перезапустите notebook. Все следующие ячейки выполняются уже в этом окружении.

In [ ]:
import os
import sys
from pathlib import Path

VENV_PY = Path(".venv") / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not VENV_PY.exists():
    !"{sys.executable}" -m venv .venv
!"{VENV_PY}" -m pip install -q -r requirements.txt
!"{VENV_PY}" -m ipykernel install --user --name avito-orient --display-name "Python (avito-orient)"

In [ ]:
import sys
from pathlib import Path

if Path(sys.prefix).resolve() != (Path.cwd() / ".venv").resolve():
    raise RuntimeError(f"Ядро не из .venv: {sys.executable}. Выберите ядро .venv и перезапустите notebook.")

sys.path.insert(0, str(Path.cwd() / "src"))

from orient import download
from orient.paths import DATA, TEST_IMAGES

## 1. Скачивание данных

Обучающих данных в задании нет, поэтому собираем свои. Метки ориентации для них получаем бесплатно: берём ровный текст и половину поворачиваем на 180°.

| Источник | Зачем |
|---|---|
| Корпуса Leipzig: русские новости, русская Википедия, английские новости | тексты для синтетических строк |
| Google Fonts (лицензия OFL), ~70 семейств с кириллицей | разные начертания для синтетики |
| Веса ImageNet из `timm` | инициализация учителя и студента |
| Cyrillic Handwriting Dataset (Kaggle) | реальные рукописные строки |
| Фото объявлений из Avito Duplicate Ads Detection (Kaggle) | реальные кропы: валидация и пул для дистилляции |

In [ ]:
download.download_corpora()

In [ ]:
missing = download.download_fonts()
print("нет в google/fonts:", missing)

In [ ]:
download.download_pretrained()

### Kaggle

Нужен API-токен: Kaggle → Settings → API → Create New Token, файл `kaggle.json` положить в `~/.kaggle/` (`chmod 600 ~/.kaggle/kaggle.json`). Для соревнования Avito ещё нужно принять правила на его странице.

Данные соревнования разрешены только для некоммерческого использования; здесь они нужны лишь для обучения и валидации.

In [ ]:
download.kaggle_download("datasets", "constantinwerner/cyrillic-handwriting-dataset", DATA / "handwriting")
# Если имя архива не подошло, список файлов: !kaggle competitions files avito-duplicate-ads-detection
download.kaggle_download("competitions", "avito-duplicate-ads-detection", DATA / "photos", file="Images_0.zip")

In [ ]:
download.summary()